### Goal
Build a DPD for the PA, using the LSTM model of the PA (`best_model.pt`) as a differentiable digital twin.

Target: `PA(DPD(x)) ≈ G·x`

#### Etapa 1 — Load the PA digital twin and validate it
1. Same data and normalization as the training
2. LSTM frozen (weights don't change, gradient still flows to the input)
3. Check that it reproduces the measured PA, and how it behaves beyond the training range

#### 1.1 Setup

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from pa_model import Dataset, DATA_PROCESSED, RESULTS
from lstm import PaLSTM
from metrics.metrics import EVM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print("device:", device)

#### 1.2 Data and normalization (same as training)

In [ ]:
ds = Dataset.from_csv(DATA_PROCESSED / "expData.csv")
train, val, test = ds.split()                 # 60/20/20, sem embaralhar
_, scaler_x, scaler_y = train.normalize()     # StandardScaler ajustado SÓ no treino

def to_tensor(z, scaler):
    """complexo (N,) em unidades físicas -> tensor (N, 2) [I, Q] normalizado"""
    iq = np.stack([z.real, z.imag], axis=1)
    return torch.tensor(scaler.transform(iq), dtype=torch.float32, device=device)

def to_complex(t, scaler):
    """tensor (N, 2) normalizado -> complexo (N,) em unidades físicas"""
    iq = scaler.inverse_transform(t.detach().cpu().numpy().astype(np.float64))
    return iq[:, 0] + 1j * iq[:, 1]

#### 1.3 Loading the frozen LSTM

In [ ]:
params = json.load(open(RESULTS / "best_params.json"))["params"]
W = params["ws"]                               # janela = 10

pa = PaLSTM(2, params["hs"], params["nl"], params["dr"]).to(device)
pa.load_state_dict(torch.load(RESULTS / "best_model.pt", map_location=device, weights_only=True))
pa.eval()
for p in pa.parameters():
    p.requires_grad_(False)                    # congela: o treino do DPD nunca altera o PA

#### 1.4 Passing a signal through the PA model

In [ ]:
def pa_forward(u):
    """u: (N, 2) normalizado -> (N-W+1, 2) normalizado.
    A saída k corresponde à amostra k+W-1 (último instante de cada janela)."""
    windows = u.unfold(0, W, 1).transpose(1, 2)    # (N-W+1, W, 2)
    return pa(windows)

#### 1.5 Validation: does the model reproduce the measured PA?

In [ ]:
x_test = to_tensor(test.x, scaler_x)
with torch.no_grad():
    y_hat = to_complex(pa_forward(x_test), scaler_y)

x_ref, y_ref = test.x[W-1:], test.y[W-1:]       # alinhados com a saída de pa_forward
print(f"EVM do PA_LSTM no teste: {EVM().compute(y_ref, y_hat):.3f} %")

#### 1.6 Extrapolation: how does the model behave beyond the training range?

In [ ]:
alpha = 1.3                                      # entrada 30% maior que a medida
with torch.no_grad():
    y_ext = to_complex(pa_forward(to_tensor(alpha * test.x, scaler_x)), scaler_y)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.abs(x_ref), np.abs(y_ref), '.', ms=2, alpha=.2, label="PA medido")
ax.plot(np.abs(x_ref), np.abs(y_hat), '.', ms=2, alpha=.2, label="PA_LSTM")
ax.plot(alpha*np.abs(x_ref), np.abs(y_ext), '.', ms=2, alpha=.2, label=f"PA_LSTM com x × {alpha}")
ax.axvline(np.abs(train.x).max(), color="k", ls="--", lw=1, label="máx |x| do treino")
ax.set_xlabel("|x|"); ax.set_ylabel("|y|"); ax.set_title("AM/AM — validação e extrapolação")
ax.grid(True, alpha=.3); ax.legend(markerscale=8)
plt.show()